# VERITAS-V: Industrial 6-Phase Dual-Model Training & Alignment Pipeline

This notebook trains and aligns both foundation models for **VERITAS-V**:
1. **Model 1 (`Veritas-Captioner`)**: QLoRA fine-tuning on `Qwen2.5-VL-3B-Instruct` for dense multimodal temporal grounding.
2. **Model 2 (`Veritas-CausalVerifier`)**: Multi-stage SFT + Direct Preference Optimization (DPO) on `Llama-3.1-8B-Instruct` for causal inference and hallucination verification.
3. **Fusion & Packaging**: Direct LoRA adapter weight fusion into standalone Safetensors with quantization metadata.
4. **Adversarial Red-Teaming**: Formal certification against 5 adversarial attack suites.

In [ ]:
# 1. Install Training & Alignment Dependencies
!pip install -q torch transformers datasets peft accelerate bitsandbytes trl qwen-vl-utils

In [ ]:
# 2. Verify GPU Acceleration Environment
import torch
print("GPU Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected!")
print("VRAM Available (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else 0)

In [ ]:
# 3. Phase 1 & 2: Generate Scaled Multi-Domain SFT & DPO Preference Datasets
!python scripts/training/expand_multimodal_dataset.py
!python scripts/training/build_dpo_dataset.py
!python scripts/training/validate_and_split_data.py

In [ ]:
# 4. Phase 3A: Fine-Tune Model 1 (Veritas-Captioner VLM)
!python scripts/training/train_captioner.py --epochs 3 --batch-size 2 --lr 2e-4

In [ ]:
# 5. Phase 3B: Fine-Tune Model 2 (Veritas-CausalVerifier SFT)
!python scripts/training/train_reasoner.py --epochs 3 --batch-size 4 --lr 2e-4

In [ ]:
# 6. Phase 3C: Direct Preference Optimization (DPO Alignment)
!python scripts/training/train_dpo_verifier.py --epochs 2 --beta 0.1 --lr 5e-6

In [ ]:
# 7. Phase 4: Merge LoRA Adapters into Standalone Backbones
!python scripts/training/merge_and_export.py

In [ ]:
# 8. Phase 5: Adversarial Red-Teaming & Calibration Certification
!python scripts/training/stress_test_and_redteam.py
!python scripts/run_all_benchmarks.py

In [ ]:
# 9. Optional: Run Complete 6-Phase Master Orchestrator in 1 Command
!python scripts/training/orchestrate_training.py --full-gpu